<a href="https://colab.research.google.com/github/Meeran11-A/CSA-1724-AI/blob/main/Exp_06_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from collections import deque


def reflex_vacuum_agent(location: str, status: str) -> str:
    """Simple reflex agent based on current percept only."""
    if status == "Dirty":
        return "Suck"
    elif location == "A":
        return "Right"
    else:
        return "Left"


def run_reflex_simulation(start_loc: str, status_a: str, status_b: str, max_steps: int = 8):
    """Runs the reflex agent and logs each step."""
    env = {"A": status_a.capitalize(), "B": status_b.capitalize()}
    loc = start_loc.upper()

    print(f"--- Reflex Agent Simulation (Start: {loc}, A: {env['A']}, B: {env['B']}) ---")
    for step in range(1, max_steps + 1):
        action = reflex_vacuum_agent(loc, env[loc])
        print(f"Step {step}: Percept=({loc}, {env[loc]}) -> Action={action}")

        if action == "Suck":
            env[loc] = "Clean"
        elif action == "Right":
            loc = "B"
        elif action == "Left":
            loc = "A"

        if env["A"] == "Clean" and env["B"] == "Clean":
            print(f"All rooms clean after {step} step(s). State: A={env['A']}, B={env['B']}\n")
            return
    print("Max steps reached.\n")


def solve_vacuum_bfs(start_loc: str, status_a: str, status_b: str):
    """
    Finds the shortest action sequence to make both Room A and Room B Clean.
    State representation: (agent_loc, status_A, status_B)
    """
    start = (start_loc.upper(), status_a.capitalize(), status_b.capitalize())

    if start[1] == "Clean" and start[2] == "Clean":
        return []

    queue = deque([(start, [])])
    visited = {start}

    while queue:
        (loc, sa, sb), plan = queue.popleft()

        # Candidate transitions: (action, next_state)
        transitions = []
        if loc == "A":
            new_sa = "Clean" if sa == "Dirty" else sa
            transitions.append(("Suck", ("A", new_sa, sb)))
            transitions.append(("Right", ("B", sa, sb)))
        else:
            new_sb = "Clean" if sb == "Dirty" else sb
            transitions.append(("Suck", ("B", sa, new_sb)))
            transitions.append(("Left", ("A", sa, sb)))

        for action, nxt in transitions:
            if nxt[1] == "Clean" and nxt[2] == "Clean":
                return plan + [(action, nxt)]
            if nxt not in visited:
                visited.add(nxt)
                queue.append((nxt, plan + [(action, nxt)]))

    return None


if __name__ == "__main__":
    # Example 1: Agent starts at A, both rooms Dirty
    run_reflex_simulation("A", "Dirty", "Dirty")

    bfs_plan = solve_vacuum_bfs("A", "Dirty", "Dirty")
    print("--- Shortest Plan via BFS (Start: A, A: Dirty, B: Dirty) ---")
    for idx, (act, state) in enumerate(bfs_plan, start=1):
        print(f"Action {idx}: {act:<5} -> New State: Loc={state[0]}, A={state[1]}, B={state[2]}")

--- Reflex Agent Simulation (Start: A, A: Dirty, B: Dirty) ---
Step 1: Percept=(A, Dirty) -> Action=Suck
Step 2: Percept=(A, Clean) -> Action=Right
Step 3: Percept=(B, Dirty) -> Action=Suck
All rooms clean after 3 step(s). State: A=Clean, B=Clean

--- Shortest Plan via BFS (Start: A, A: Dirty, B: Dirty) ---
Action 1: Suck  -> New State: Loc=A, A=Clean, B=Dirty
Action 2: Right -> New State: Loc=B, A=Clean, B=Dirty
Action 3: Suck  -> New State: Loc=B, A=Clean, B=Clean
